In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

raiz_projeto = str(Path().resolve().parent)
if raiz_projeto not in sys.path:
    sys.path.append(raiz_projeto)

from src.data_loader import SEEDS_TREINO, fixar_sementes, preparar_dados_pytorch
from src.metrics import calcular_metricas_classificacao, calcular_metricas_regressao
from src.models import MLP
from src.train import treinar

os.makedirs("../reports/tables", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
# 1. Carregamento dos dados (mesma divisão 80/20 e random_state=42)
fixar_sementes(42)
loaders_reg, scaler_X_reg, scaler_y_reg, n_features_reg = preparar_dados_pytorch(
    caminho_csv="../data/processed/dataset_regression.csv",
    target_col="term_gpa",
    tipo="reg",
    batch_size=32,
)

loaders_cls, scaler_X_cls, _, n_features_cls = preparar_dados_pytorch(
    caminho_csv="../data/processed/dataset_classification.csv",
    target_col="gpa_category_encoded",
    tipo="class",
    batch_size=32,
)

In [3]:
def obter_previsoes(modelo: nn.Module, dataset, tarefa: str):
    """Gera previsões em modo eval() sobre um dataset inteiro."""
    dl = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    modelo.eval()
    with torch.inference_mode():
        xb, yb = next(iter(dl))
        saida = modelo(xb.to(device)).cpu().numpy().ravel()
        reais = yb.numpy().ravel()

    if tarefa == "regressao":
        preds = scaler_y_reg.inverse_transform(saida.reshape(-1, 1)).ravel()
        reais_orig = scaler_y_reg.inverse_transform(reais.reshape(-1, 1)).ravel()
        return reais_orig, preds
    else:
        probs = 1.0 / (1.0 + np.exp(-np.clip(saida, -50, 50)))
        return reais.astype(int), probs

In [4]:
def avaliar_treino_e_teste(tarefa: str, nome_modelo: str, hidden: tuple, lr: float):
    """Treina nas sementes e calcula métricas no treino, teste e gap de generalização."""
    loaders = loaders_reg if tarefa == "regressao" else loaders_cls
    in_dim = n_features_reg if tarefa == "regressao" else n_features_cls
    loss_fn = nn.MSELoss() if tarefa == "regressao" else nn.BCEWithLogitsLoss()

    registros_sementes = []
    melhor_modelo = None
    melhor_val = float("inf")
    artefatos_melhor = None

    for seed in SEEDS_TREINO:
        fixar_sementes(seed)
        modelo = MLP(in_dim=in_dim, out_dim=1, hidden=hidden, activation="ReLU", dropout=0.0)
        otimizador = torch.optim.Adam(modelo.parameters(), lr=lr)

        hist = treinar(
            modelo=modelo,
            dl_tr=loaders["train"],
            dl_val=loaders["val"],
            loss_fn=loss_fn,
            otimizador=otimizador,
            n_epocas=200,
            paciencia=20,
            device=device,
        )

        y_tr, pred_tr = obter_previsoes(modelo, loaders["train"].dataset, tarefa)
        y_te, pred_te = obter_previsoes(modelo, loaders["test"].dataset, tarefa)

        if tarefa == "regressao":
            m_tr = calcular_metricas_regressao(y_tr, pred_tr)
            m_te = calcular_metricas_regressao(y_te, pred_te)
            reg = {
                **{f"{k}_Treino": v for k, v in m_tr.items()},
                **{f"{k}_Teste": v for k, v in m_te.items()},
                "Gap_RMSE (Teste-Treino)": m_te["RMSE"] - m_tr["RMSE"],
                "Gap_R2 (Treino-Teste)": m_tr["R2"] - m_te["R2"],
            }
        else:
            m_tr = calcular_metricas_classificacao(y_tr, pred_tr)
            m_te = calcular_metricas_classificacao(y_te, pred_te)
            reg = {
                **{f"{k}_Treino": v for k, v in m_tr.items()},
                **{f"{k}_Teste": v for k, v in m_te.items()},
                "Gap_F1 (Treino-Teste)": m_tr["F1"] - m_te["F1"],
                "Gap_Acc (Treino-Teste)": m_tr["Acuracia"] - m_te["Acuracia"],
            }

        registros_sementes.append(reg)

        if min(hist["val"]) < melhor_val:
            melhor_val = min(hist["val"])
            melhor_modelo = modelo
            artefatos_melhor = (y_te, pred_te)

    df_s = pd.DataFrame(registros_sementes)
    resumo = {
        "Tarefa": tarefa,
        "Modelo": nome_modelo,
        "Arquitetura": str(hidden),
        "LR": lr,
        "Parametros": melhor_modelo.contar_parametros(),
    }
    for col in df_s.columns:
        resumo[f"{col}_media"] = float(df_s[col].mean())
        resumo[f"{col}_std"] = float(df_s[col].std())
        resumo[col] = f"{df_s[col].mean():.4f} ± {df_s[col].std():.4f}"

    return resumo, melhor_modelo, artefatos_melhor

In [5]:
# 2. Execução para Baseline e Modelo Otimizado em ambas as tarefas
res_reg_base, _, _ = avaliar_treino_e_teste("regressao", "MLP Baseline", (32, 32), 1e-3)
res_reg_opt, mod_reg_opt, dados_te_reg = avaliar_treino_e_teste("regressao", "MLP Otimizada", (128, 128), 1e-2)

res_cls_base, _, _ = avaliar_treino_e_teste("classificacao", "MLP Baseline", (32, 32), 1e-3)
res_cls_opt, mod_cls_opt, dados_te_cls = avaliar_treino_e_teste("classificacao", "MLP Otimizada", (128, 128), 1e-3)

# 3. Exportação da tabela-resumo final
df_resumo_final = pd.DataFrame([res_reg_base, res_reg_opt, res_cls_base, res_cls_opt])
df_resumo_final.to_csv("../reports/tables/resumo_final.csv", index=False)

print("=== TABELA FINAL — REGRESSÃO (TREINO vs. TESTE E GAP) ===")
cols_reg = ["Modelo", "Arquitetura", "LR", "RMSE_Treino", "RMSE_Teste", "Gap_RMSE (Teste-Treino)", "MAE_Teste", "R2_Treino", "R2_Teste", "MAPE (%)_Teste"]
display(df_resumo_final[df_resumo_final["Tarefa"] == "regressao"][cols_reg])

print("\n=== TABELA FINAL — CLASSIFICAÇÃO (TREINO vs. TESTE E GAP) ===")
cols_cls = ["Modelo", "Arquitetura", "LR", "Acuracia_Treino", "Acuracia_Teste", "F1_Treino", "F1_Teste", "Gap_F1 (Treino-Teste)", "Precisao_Teste", "Recall_Teste", "ROC_AUC_Teste", "PR_AUC_Teste"]
display(df_resumo_final[df_resumo_final["Tarefa"] == "classificacao"][cols_cls])

=== TABELA FINAL — REGRESSÃO (TREINO vs. TESTE E GAP) ===


,Modelo,Arquitetura,LR,RMSE_Treino,RMSE_Teste,Gap_RMSE (Teste-Treino),MAE_Teste,R2_Treino,R2_Teste,MAPE (%)_Teste
0,MLP Baseline,"(32, 32)",0.001,0.3409 ± 0.0038,1.6756 ± 0.0047,1.3348 ± 0.0038,1.6567 ± 0.0053,0.4698 ± 0.0117,-48.0757 ± 0.2764,32.5849 ± 0.0992
1,MLP Otimizada,"(128, 128)",0.010,0.3413 ± 0.0126,1.7292 ± 0.0390,1.3879 ± 0.0490,1.7008 ± 0.0403,0.4679 ± 0.0387,-51.2850 ± 2.3441,33.4960 ± 0.7846



=== TABELA FINAL — CLASSIFICAÇÃO (TREINO vs. TESTE E GAP) ===


,Modelo,Arquitetura,LR,Acuracia_Treino,Acuracia_Teste,F1_Treino,F1_Teste,Gap_F1 (Treino-Teste),Precisao_Teste,Recall_Teste,ROC_AUC_Teste,PR_AUC_Teste
2,MLP Baseline,"(32, 32)",0.001,0.7615 ± 0.0063,0.7762 ± 0.0087,0.7695 ± 0.0047,0.7880 ± 0.0070,-0.0185 ± 0.0075,0.7596 ± 0.0122,0.8187 ± 0.0086,0.8484 ± 0.0062,0.8581 ± 0.0050
3,MLP Otimizada,"(128, 128)",0.001,0.7500 ± 0.0071,0.7714 ± 0.0118,0.7603 ± 0.0064,0.7864 ± 0.0100,-0.0261 ± 0.0130,0.7490 ± 0.0168,0.8281 ± 0.0191,0.8489 ± 0.0039,0.8572 ± 0.0049
